- japonica: NH001 NH002 NH003 NH005
- indica: NH006 NH007 NH008 NH009
- genome: /mnt/rice/default/Workspace/yangdong/ai4research/rice_server/source/rice_mut/osa1_r7.asm.ch.fa
- annotation: /mnt/rice/default/Workspace/yangdong/ai4research/rice_server/source/rice_mut/osa1_r7.all_models.gff3
- leaf_ck: /mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/analysis/six_context_tmm_bigwig_links/leaf_ck
- leaf_salt: /mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/analysis/six_context_tmm_bigwig_links/leaf_salt


1. **峰值位点（Lead SNP）**：位于 **5号染色体（Chr. 5）的 28,351,660 bp** 处，该位点被命名为 **qSTS5**（Salt Tolerance Site 5）。

2. **初步候选区间（用于初筛）**：研究优先关注了该峰值位点 **上游和下游各 200 kb** 的区域（即 Chr5: 28,151,660 ~ 28,551,660），在此范围内初步筛选出了 **97个** 候选基因。

3. **核心精细定位区间（用于多组学交集）**：在进一步的整合分析（结合DEGs、eQTL和GWAS）中，这97个候选基因被定位在峰值附近一个更精确的基因组区域内，原文描述为 **“S46.49 Kb”**（从上下文推测“S”应为“~”约等于，即 **约 46.49 kb** 的区间）。随后，通过基于非同义突变的单倍型分析，将候选基因进一步缩小至 **31个**。

**总结**：GWAS信号的最显著峰值在 **5号染色体 28,351,660 bp**，研究主要围绕该位点 **±200 kb** 的区间进行候选基因挖掘，最终通过多组学数据将核心区间聚焦在峰值附近 **约46.49 kb** 的范围内。

In [1]:
import gzip
import os
import pandas as pd

# 参数设置
gff_path = "/mnt/rice/default/Workspace/yangdong/ai4research/rice_server/source/rice_mut/osa1_r7.all_models.gff3"
fasta_path = "/mnt/rice/default/Workspace/yangdong/ai4research/rice_server/source/rice_mut/osa1_r7.asm.ch.fa"

target_chroms = {"Chr5", "chr5", "chr05", "5"}
region_start = 28051660
region_end = 28651660

def parse_gff_attributes(attr_str):
    """解析 GFF3 的 Ninth column (Attributes)"""
    attrs = {}
    for item in attr_str.strip().split(";"):
        if "=" in item:
            k, v = item.split("=", 1)
            attrs[k] = v
    return attrs

def get_genes_in_region(gff_file, target_chroms, start, end):
    genes = []
    open_fn = gzip.open if gff_file.endswith(".gz") else open
    
    with open_fn(gff_file, "rt") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split("\t")
            if len(parts) < 9:
                continue
            
            chrom, source, feature_type, g_start, g_end, score, strand, phase, attr = parts
            
            # 判断染色体及类型 (gene 或 mRNA)
            if chrom in target_chroms and feature_type in ["gene"]:
                g_start = int(g_start)
                g_end = int(g_end)
                
                # 判断区间重叠 (Start1 <= End2 AND End1 >= Start2)
                if g_start <= end and g_end >= start:
                    attrs = parse_gff_attributes(attr)
                    gene_id = attrs.get("ID", attrs.get("Name", "Unknown"))
                    genes.append({
                        "id": gene_id,
                        "chrom": chrom,
                        "start": g_start,
                        "end": g_end,
                        "strand": strand,
                        "feature_type": feature_type,
                        "attributes": attrs
                    })
    return genes

def extract_fasta_sequence(fasta_file, chroms, start, end):
    """读取指定区域的基因组序列"""
    seq_chunks = []
    current_chrom = None
    open_fn = gzip.open if fasta_file.endswith(".gz") else open
    
    with open_fn(fasta_file, "rt") as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                current_chrom = line[1:].split()[0]
            elif current_chrom in chroms:
                seq_chunks.append(line)
                
    full_seq = "".join(seq_chunks)
    # 提取切片 (1-based coordinate 转换为 0-based python slice)
    region_seq = full_seq[start - 1 : end]
    return region_seq

# 1. 获取基因列表
genes = get_genes_in_region(gff_path, target_chroms, region_start, region_end)

print(f"在 Chr5:{region_start:,}~{region_end:,} 区域共找到 {len(genes)} 个基因/转录本功能特征：\n")
# print(f"{'ID':<25} {'Type':<8} {'Start':<12} {'End':<12} {'Strand':<6}")
# print("-" * 65)

# for g in genes:
#     print(f"{g['id']:<25} {g['feature_type']:<8} {g['start']:<12} {g['end']:<12} {g['strand']:<6}")

# 2. (可选) 提取该区域的基因组 FASTA 序列
region_dna = extract_fasta_sequence(fasta_path, target_chroms, region_start, region_end)
print(f"\n区域序列长度: {len(region_dna)} bp")

df = pd.DataFrame(genes)

# 可选：如果包含 attributes 字典列，提取或去除复杂属性列以保持表格整洁
if "attributes" in df.columns:
    # 例如提取 Description/Name 属性，然后删掉原始 attributes 字典
    df["Name"] = df["attributes"].apply(lambda x: x.get("Name", x.get("ID", "")))
    df["Description"] = df["attributes"].apply(lambda x: x.get("description", x.get("Note", "")))
    df = df.drop(columns=["attributes"])

# 保存为 CSV 文件
output_csv = f"Chr5_{region_start}_{region_end}_genes.csv"
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"已成功导出 {len(df)} 条记录到 CSV 文件: {output_csv}")

在 Chr5:28,051,660~28,651,660 区域共找到 107 个基因/转录本功能特征：


区域序列长度: 600001 bp
已成功导出 107 条记录到 CSV 文件: Chr5_28051660_28651660_genes.csv


In [2]:
df.head()

,id,chrom,start,end,strand,feature_type,Name,Description
0,LOC_Os05g48900,Chr5,28051693,28052945,+,gene,LOC_Os05g48900,fasciclin%20domain%20containing%20protein%2C%2...
1,LOC_Os05g48910,Chr5,28056519,28063085,+,gene,LOC_Os05g48910,retrotransposon%20protein%2C%20putative%2C%20u...
2,LOC_Os05g48920,Chr5,28063465,28064118,+,gene,LOC_Os05g48920,retrotransposon%20protein%2C%20putative%2C%20u...
3,LOC_Os05g48930,Chr5,28067838,28068782,+,gene,LOC_Os05g48930,OsGrx_S2%20-%20glutaredoxin%20subgroup%20III%2...
4,LOC_Os05g48940,Chr5,28071956,28076493,-,gene,LOC_Os05g48940,expressed%20protein


In [3]:
import os
import re

# 1. 样本列表定义
japonica = ["NH001", "NH002", "NH003", "NH005", "NH013"]
indica = ["NH006", "NH007", "NH008", "NH009", "NH014"]
all_samples = japonica + indica

# 2. 目录路径定义
dirs = {
    "genome": "/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/genome",
    "annotation": "/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/genome_os7",
    "leaf_ck": "/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/bw/leaf_ck",
    "leaf_salt": "/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/bw/leaf_salt"
}

# 3. 初始化结果字典
sample_files = {sample: [] for sample in all_samples}

# 4. 遍历目录并匹配绝对路径
for condition, dir_path in dirs.items():
    if not os.path.exists(dir_path):
        print(f"警告: 路径不存在 -> {dir_path}")
        continue
        
    for root, _, files in os.walk(dir_path):
        for file in files:
            # 仅匹配 .bw 或 .bigwig 文件（可根据实际后缀调整）
            if file.endswith((".fasta", ".gff3", ".bw", ".bigwig")):
                full_path = os.path.abspath(os.path.join(root, file))
                
                # 遍历样本名称进行正则匹配
                for sample in all_samples:
                    # 只要文件名包含 sample，且 sample 后面跟的是非字母数字字符（如 _、.）或结尾，即可匹配
                    pattern = rf"{sample}(?=[^a-zA-Z0-9]|$)"
                    if re.search(pattern, file):
                        sample_files[sample].append(full_path)

# 打印最终构建的字典
import pprint
pprint.pprint(sample_files)

{'NH001': ['/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/genome/NH001.fasta',
           '/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/genome_os7/NH001.os7.gff3',
           '/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/bw/leaf_ck/NH001.bw',
           '/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/bw/leaf_salt/NH001.bw'],
 'NH002': ['/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/genome/NH002.fasta',
           '/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/genome_os7/NH002.os7.gff3',
           '/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/bw/leaf_ck/NH002.bw',
           '/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/bw/leaf_salt/NH002.bw'],
 'NH003': ['/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/genome/NH003.fasta',
           '/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/genome_os7/NH003.

In [4]:
import os
import pyBigWig
import pandas as pd
import numpy as np

# 1. 目标基因集 (替换为你的基因列表)
gene_list = set(df['id'])  # 示例基因集

def parse_gff_genes(gff_path, target_genes):
    """解析 GFF3，提取匹配基因的坐标信息"""
    gene_coords = {}
    if not os.path.exists(gff_path):
        return gene_coords
        
    with open(gff_path, 'r') as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split('\t')
            if len(parts) < 9 or parts[2] != 'gene':
                continue
            
            chrom = parts[0]
            start, end = int(parts[3]) - 1, int(parts[4])  # 转为 0-based 坐标
            attr = parts[8]
            
            # 解析 ID 或 Name 属性
            gene_id = None
            for item in attr.split(';'):
                if item.startswith('ID=') or item.startswith('Name='):
                    val = item.split('=')[1]
                    base_val = val.split('.')[0]  # 兼容可能的版本号后缀
                    if val in target_genes:
                        gene_id = val
                        break
                    elif base_val in target_genes:
                        gene_id = base_val
                        break
                        
            if gene_id:
                gene_coords[gene_id] = {
                    'chrom': chrom,
                    'start': start,
                    'end': end
                }
    return gene_coords

def get_mean_coverage(bw_path, chrom, start, end):
    """从 BigWig 提取直接对应染色体区间的平均覆盖度信号"""
    if not os.path.exists(bw_path):
        return 0.0

    bw = pyBigWig.open(bw_path)
    chroms = bw.chroms()
    
    if chrom not in chroms:
        bw.close()
        return 0.0

    chrom_len = chroms[chrom]
    real_start = max(0, min(start, chrom_len))
    real_end = max(0, min(end, chrom_len))
    
    if real_start >= real_end:
        bw.close()
        return 0.0

    # 获取区间平均信号 (Mean Coverage)
    val = bw.stats(chrom, real_start, real_end, type="mean")[0]
    bw.close()
    return float(val) if val is not None else 0.0

In [5]:
# 2. 统计主循环
results = []

for sample, files in sample_files.items():
    fasta_file, gff_file, ck_bw, salt_bw = files[0], files[1], files[2], files[3]
    
    # 解析重叠/匹配的基因坐标
    overlap_genes = parse_gff_genes(gff_file, gene_list)
    print(f"Sample [{sample}]: 匹配目标基因数 = {len(overlap_genes)} / {len(gene_list)}")
    
    for gene_id, pos in overlap_genes.items():
        chrom, start, end = pos['chrom'], pos['start'], pos['end']
        
        # 直接按一致的染色体名提取信号
        ck_signal = get_mean_coverage(ck_bw, chrom, start, end)
        salt_signal = get_mean_coverage(salt_bw, chrom, start, end)
        
        # 计算差异表达 (添加 1e-5 伪计数防止除零)
        fold_change = (salt_signal + 1e-5) / (ck_signal + 1e-5)
        log2_fc = np.log2(fold_change)
        
        results.append({
            'Sample': sample,
            'Gene_ID': gene_id,
            'Chrom': chrom,
            'Start': start,
            'End': end,
            'CK_Signal': round(ck_signal, 4),
            'Salt_Signal': round(salt_signal, 4),
            'FoldChange_Salt_vs_CK': round(fold_change, 4),
            'Log2_FC': round(log2_fc, 4)
        })

# 3. 汇总与导出
df_res = pd.DataFrame(results)
print("\n=== 分析结果预览 ===")
print(df_res.head(10))

Sample [NH001]: 匹配目标基因数 = 106 / 107
Sample [NH002]: 匹配目标基因数 = 107 / 107
Sample [NH003]: 匹配目标基因数 = 107 / 107
Sample [NH005]: 匹配目标基因数 = 107 / 107
Sample [NH013]: 匹配目标基因数 = 107 / 107
Sample [NH006]: 匹配目标基因数 = 105 / 107
Sample [NH007]: 匹配目标基因数 = 107 / 107
Sample [NH008]: 匹配目标基因数 = 106 / 107
Sample [NH009]: 匹配目标基因数 = 106 / 107
Sample [NH014]: 匹配目标基因数 = 106 / 107

=== 分析结果预览 ===
  Sample         Gene_ID  Chrom     Start       End  CK_Signal  Salt_Signal  \
0  NH001  LOC_Os05g49550  Chr12   9275262   9279152     0.0000       0.0037   
1  NH001  LOC_Os05g49540   Chr2  28216146  28221405     0.0000       0.0000   
2  NH001  LOC_Os05g48920   Chr3  14968524  14969178     0.0000       0.0000   
3  NH001  LOC_Os05g49090   Chr5  18076413  18076725     0.0000       0.0000   
4  NH001  LOC_Os05g48900   Chr5  28140385  28141637     0.1827       2.0621   
5  NH001  LOC_Os05g48910   Chr5  28145210  28151765     0.0057       0.0130   
6  NH001  LOC_Os05g48930   Chr5  28167751  28168689     3.3257       3.

In [6]:
df_res.to_csv("gene_set_expression_diff_salt_vs_ck.csv", index=False)

In [7]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

# 假设 df_res 已经按之前代码生成
# df_res 列名包含: ['Sample', 'Gene_ID', 'CK_Signal', 'Salt_Signal']

gene_stat_results = []

# 按 Gene_ID 分组计算组间显著性
for gene_id, group in df_res.groupby('Gene_ID'):
    ck_vals = group['CK_Signal'].values
    salt_vals = group['Salt_Signal'].values

    # 计算全局均值 (Mean Signal)
    mean_ck = np.mean(ck_vals)
    mean_salt = np.mean(salt_vals)

    # 计算全局差异倍数 Log2FC (加入 1e-5 伪计数)
    overall_fold_change = (mean_salt + 1e-5) / (mean_ck + 1e-5)
    log2_fc = np.log2(overall_fold_change)

    # 统计检验 (样本数 >= 3 时使用 Wilcoxon 秩和检验/Mann-Whitney U 检验；配对设计可用 Wilcoxon signed-rank)
    # 如果样本量较小或信号完全全等，捕获 Exception 防止中断
    try:
        # 针对配对样本 (每个样本有对应的 CK 和 Salt)
        if len(ck_vals) >= 3 and not np.all(ck_vals == salt_vals):
            stat, p_val = stats.wilcoxon(salt_vals, ck_vals)
            # 或者非配对 Mann-Whitney U 检验:
            # stat, p_val = stats.mannwhitneyu(salt_vals, ck_vals, alternative='two-sided')
        else:
            p_val = 1.0
    except Exception:
        p_val = 1.0

    gene_stat_results.append({
        'Gene_ID': gene_id,
        'Mean_CK': round(mean_ck, 4),
        'Mean_Salt': round(mean_salt, 4),
        'Log2FC': round(log2_fc, 4),
        'pvalue': p_val if not np.isnan(p_val) else 1.0,
    })

# 转换为 DataFrame
df_stat = pd.DataFrame(gene_stat_results)

# 多重假设检验校正 (BH 方法计算 p.adj)
if not df_stat.empty:
    _, p_adj, _, _ = multipletests(df_stat['pvalue'], method='fdr_bh')
    df_stat['p.adj'] = np.round(p_adj, 6)
    df_stat['pvalue'] = np.round(df_stat['pvalue'], 6)

    # 标注上调/下调显著性分类 (Sig_State)
    conditions = [
        (df_stat['p.adj'] < 0.05) & (df_stat['Log2FC'] >= 1.0),
        (df_stat['p.adj'] < 0.05) & (df_stat['Log2FC'] <= -1.0),
    ]
    choices = ['Up', 'Down']
    df_stat['Sig_State'] = np.select(conditions, choices, default='Not Sig')

print("=== 基因组间显著性差异分析结果 (DEG Summary) ===")
print(df_stat.head(15))

# 导出显著性结果
df_stat.to_csv('gene_diff_expression_stats.csv', index=False)

=== 基因组间显著性差异分析结果 (DEG Summary) ===
           Gene_ID  Mean_CK  Mean_Salt  Log2FC    pvalue     p.adj Sig_State
0   LOC_Os05g48900   0.1529     2.3952  3.9690  0.001953  0.009952        Up
1   LOC_Os05g48910   0.0012     0.0084  2.7404  0.046875  0.122332   Not Sig
2   LOC_Os05g48920   0.0000     0.0076  9.5718  1.000000  1.000000   Not Sig
3   LOC_Os05g48930   2.4552     2.1355 -0.2012  0.322266  0.574707   Not Sig
4   LOC_Os05g48940   0.6153     0.6766  0.1370  0.160156  0.342734   Not Sig
5   LOC_Os05g48950   0.9015     1.4933  0.7281  0.003906  0.015480   Not Sig
6   LOC_Os05g48960   3.0303     2.9205 -0.0533  0.625000  0.941901   Not Sig
7   LOC_Os05g48970   0.5013     0.5777  0.2045  0.375000  0.657787   Not Sig
8   LOC_Os05g48980   1.1298     1.1026 -0.0351  0.769531  1.000000   Not Sig
9   LOC_Os05g48990   0.0000     0.0013  7.0112  0.250000  0.495370   Not Sig
10  LOC_Os05g49000   0.0000     0.0022  7.8074  0.250000  0.495370   Not Sig
11  LOC_Os05g49010   0.0000     0.0051  